# EPS E. coli Model — Working Notebook

This notebook works directly from the pre-saved **`eps_ecoli_model.xml`** ("the new model") — iML1515 with all three EPS pathways (cellulose, colanic acid, PNAG) already added and saved to disk. It was produced in `eps_analysis.ipynb`, in the cell that ends with `write_sbml_model(model, "eps_ecoli_model.xml")` (right after the v3 three-pathway sanity checks).

Loading this file means we don't need to re-add the three pathways from scratch in every notebook — just load and go.

**Model file:** `eps_ecoli_model.xml` (repo root, gitignored like `iML1515.xml` — must be present locally to run this notebook).

In [1]:
from cobra.io import read_sbml_model
from cobra.flux_analysis import pfba

MODEL_PATH = 'eps_ecoli_model.xml'
BIOMASS_RXN = 'BIOMASS_Ec_iML1515_core_75p37M'

model = read_sbml_model(MODEL_PATH)
print(model)
print(f"Reactions:  {len(model.reactions)}")
print(f"Metabolites: {len(model.metabolites)}")
print(f"Genes:      {len(model.genes)}")
print(f"Objective:  {model.objective.expression}")

iML1515
Reactions:  2720
Metabolites: 1882
Genes:      1516
Objective:  1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685


## Verify the three EPS pathways are present

Sanity check that all 8 EPS reactions (cellulose, colanic acid, PNAG) survived the save/reload round-trip with correct stoichiometry, and that unconstrained max flux for each still matches the previously validated values from `eps_analysis.ipynb`:
- Cellulose: 8.9260 mmol/gDW/h
- Colanic acid: 1.6618 mmol/gDW/h
- PNAG: 6.7128 mmol/gDW/h

In [2]:
eps_reactions = ['CELSYNTH', 'CELLULOSEt', 'EX_cellulose_e',
                  'COLASYNTH', 'COLAIDt', 'EX_colacid_e',
                  'PUACGAMex', 'EX_puacgam_e']

for rid in eps_reactions:
    if rid in model.reactions:
        r = model.reactions.get_by_id(rid)
        print(f"{rid:16s} FOUND  bounds=({r.lower_bound}, {r.upper_bound})  {r.reaction}")
    else:
        print(f"{rid:16s} NOT FOUND -- something is wrong, re-check eps_ecoli_model.xml")

print()
for rxn_id, label in [('EX_cellulose_e', 'Cellulose'), ('EX_colacid_e', 'Colanic acid'), ('EX_puacgam_e', 'PNAG')]:
    with model:
        model.objective = rxn_id
        sol = model.optimize()
        flux = sol.fluxes.get(rxn_id, 0.0)
        print(f"  Sanity check {label}: {flux:.4f} mmol/gDW/h")

CELSYNTH         FOUND  bounds=(0.0, 1000.0)  0.001 cdigmp_c + udpg_c --> cellulose_c + udp_c
CELLULOSEt       FOUND  bounds=(0.0, 1000.0)  cellulose_c --> cellulose_e
EX_cellulose_e   FOUND  bounds=(0.0, 1000.0)  cellulose_e --> 
COLASYNTH        FOUND  bounds=(0.0, 1000.0)  2.0 gdpfuc_c + pep_c + udcpgl_c + udpgal_c + udpglcur_c --> colacid_c + 2.0 gdp_c + pi_c + udcpp_c + 2.0 udp_c
COLAIDt          FOUND  bounds=(0.0, 1000.0)  colacid_c --> colacid_e
EX_colacid_e     FOUND  bounds=(0.0, 1000.0)  colacid_e --> 
PUACGAMex        FOUND  bounds=(0.0, 1000.0)  puacgam_p --> puacgam_e
EX_puacgam_e     FOUND  bounds=(0.0, 1000.0)  puacgam_e --> 

  Sanity check Cellulose: 8.9260 mmol/gDW/h
  Sanity check Colanic acid: 1.6618 mmol/gDW/h
  Sanity check PNAG: 6.7128 mmol/gDW/h


## Check the medium currently saved in the model — M9 or LB?

This matters because the medium is baked into the saved exchange bounds — whatever was active in the model object at the moment `write_sbml_model(...)` was called in `eps_analysis.ipynb` is what got persisted here. It won't automatically be LB just because the collaborator works in LB — we need to check, and set it explicitly if it isn't.

In [3]:
current_medium = model.medium
print(f"Number of open exchange reactions: {len(current_medium)}")
print()
for rxn_id, bound in sorted(current_medium.items()):
    print(f"  {rxn_id:16s} {bound}")

Number of open exchange reactions: 24

  EX_ca2_e         1000.0
  EX_cl_e          1000.0
  EX_co2_e         1000.0
  EX_cobalt2_e     1000.0
  EX_cu2_e         1000.0
  EX_fe2_e         1000.0
  EX_fe3_e         1000.0
  EX_glc__D_e      10.0
  EX_h2o_e         1000.0
  EX_h_e           1000.0
  EX_k_e           1000.0
  EX_mg2_e         1000.0
  EX_mn2_e         1000.0
  EX_mobd_e        1000.0
  EX_na1_e         1000.0
  EX_nh4_e         1000.0
  EX_ni2_e         1000.0
  EX_o2_e          1000.0
  EX_pi_e          1000.0
  EX_sel_e         1000.0
  EX_slnt_e        1000.0
  EX_so4_e         1000.0
  EX_tungs_e       1000.0
  EX_zn2_e         1000.0


In [4]:
# Classify: LB (per eps_media_comparison.ipynb's lb_medium) has ~20 amino acid
# exchanges open with finite (non-1000) bounds. M9 (the m9_medium_default
# convention used everywhere else in this project) has only glucose capped;
# everything else is left unconstrained (1000).
amino_acid_exchanges = ['EX_ala__L_e','EX_arg__L_e','EX_asn__L_e','EX_asp__L_e','EX_cys__L_e',
                          'EX_gln__L_e','EX_glu__L_e','EX_gly_e','EX_his__L_e','EX_ile__L_e',
                          'EX_leu__L_e','EX_lys__L_e','EX_met__L_e','EX_phe__L_e','EX_pro__L_e',
                          'EX_ser__L_e','EX_thr__L_e','EX_trp__L_e','EX_tyr__L_e','EX_val__L_e']
aa_open = [aa for aa in amino_acid_exchanges if aa in current_medium]

print(f"Amino acid exchanges open: {len(aa_open)} / {len(amino_acid_exchanges)}")
print()

if aa_open:
    print(f"LB-like: {len(aa_open)} amino acid exchanges are open -> this looks like LB medium.")
else:
    glc_bound = current_medium.get('EX_glc__D_e')
    others_unbounded = all(v == 1000.0 for k, v in current_medium.items() if k != 'EX_glc__D_e')
    if glc_bound is not None and others_unbounded:
        print(f"M9-like (default SBML convention): only glucose is capped ({glc_bound} mmol/gDW/h),")
        print("everything else is unconstrained (1000).")
        print()
        print("This is the M9 baseline convention (m9_medium_default) used throughout the project --")
        print("NOT LB. If this notebook needs LB, the medium must be explicitly set here (see next cell).")
    else:
        print("Doesn't clearly match either convention -- inspect the printed medium above manually.")

Amino acid exchanges open: 0 / 20

M9-like (default SBML convention): only glucose is capped (10.0 mmol/gDW/h),
everything else is unconstrained (1000).

This is the M9 baseline convention (m9_medium_default) used throughout the project --
NOT LB. If this notebook needs LB, the medium must be explicitly set here (see next cell).


## Set up LB medium and save `eps_ecoli_model_lb.xml`

The collaborator works exclusively in LB, so we need an LB-baked-in counterpart to `eps_ecoli_model.xml` (which — per the check above — has M9 baked in). This uses the exact same `lb_medium` dict as `eps_media_comparison.ipynb` (casein hydrolysate amino acids + yeast extract vitamins, ~40 components), so results stay consistent with the rest of the project.

This section: (1) defines `lb_medium`, (2) applies it to the in-memory model, (3) re-runs the same sanity checks and M9/LB classifier from above to confirm the switch worked, (4) saves it as a new file, and (5) reloads that new file fresh to verify the save/reload round-trip — the same verification rigor used for `eps_ecoli_model.xml` itself.

In [5]:
# LB rich medium — casein hydrolysate amino acids + yeast extract vitamins
# (exact same dict as eps_media_comparison.ipynb's lb_medium, for consistency)
lb_medium = {
    # salts / ions
    'EX_pi_e':      1000.0,
    'EX_ni2_e':     10000.0,
    'EX_so4_e':     10000.0,
    'EX_o2_e':      23.0,
    'EX_na1_e':     1000.0,
    'EX_cl_e':      1000.0,
    'EX_k_e':       1000.0,
    'EX_mg2_e':     1000.0,
    'EX_ca2_e':     1000.0,
    'EX_fe2_e':     1000.0,
    'EX_mn2_e':     1000.0,
    'EX_zn2_e':     1000.0,
    'EX_cu2_e':     1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_mobd_e':    1000.0,
    # amino acids (casein hydrolysate, proportional to composition)
    'EX_ala__L_e':  39.89,
    'EX_arg__L_e':  58.51,
    'EX_asn__L_e':  26.60,
    'EX_asp__L_e':  45.21,
    'EX_cys__L_e':  13.30,
    'EX_gln__L_e':  50.53,
    'EX_glu__L_e':  50.53,
    'EX_gly_e':     63.83,
    'EX_his__L_e':  26.60,
    'EX_ile__L_e':  42.55,
    'EX_leu__L_e':  95.74,
    'EX_lys__L_e':  63.83,
    'EX_met__L_e':  31.91,
    'EX_phe__L_e':  47.87,
    'EX_pro__L_e':  34.57,
    'EX_ser__L_e':  71.81,
    'EX_thr__L_e':  37.23,
    'EX_trp__L_e':  10.64,
    'EX_tyr__L_e':  39.89,
    'EX_val__L_e':  42.55,
    # vitamins (yeast extract)
    'EX_thm_e':     1000.0,
    'EX_nac_e':     1000.0,
    'EX_pnto__R_e': 1000.0,
    'EX_pydx_e':    1000.0,
    'EX_btn_e':     1000.0,
}

model.medium = lb_medium
print(f"lb_medium has {len(lb_medium)} components")
print(f"Model medium now has {len(model.medium)} open exchange reactions")
print(f"Growth rate under LB: {model.slim_optimize():.4f} h⁻¹")

lb_medium has 40 components
Model medium now has 40 open exchange reactions
Growth rate under LB: 1.9527 h⁻¹


In [6]:
# Re-run the EPS sanity checks under LB (unconstrained max flux per pathway)
for rxn_id, label in [('EX_cellulose_e', 'Cellulose'), ('EX_colacid_e', 'Colanic acid'), ('EX_puacgam_e', 'PNAG')]:
    with model:
        model.objective = rxn_id
        sol = model.optimize()
        flux = sol.fluxes.get(rxn_id, 0.0)
        print(f"  Sanity check {label} (LB, unconstrained): {flux:.4f} mmol/gDW/h")

# Re-run the same M9/LB classifier used above, to confirm the switch worked
current_medium = model.medium
aa_open = [aa for aa in [
    'EX_ala__L_e','EX_arg__L_e','EX_asn__L_e','EX_asp__L_e','EX_cys__L_e',
    'EX_gln__L_e','EX_glu__L_e','EX_gly_e','EX_his__L_e','EX_ile__L_e',
    'EX_leu__L_e','EX_lys__L_e','EX_met__L_e','EX_phe__L_e','EX_pro__L_e',
    'EX_ser__L_e','EX_thr__L_e','EX_trp__L_e','EX_tyr__L_e','EX_val__L_e',
] if aa in current_medium]
print()
print(f"Amino acid exchanges open: {len(aa_open)} / 20")
print("LB-like: confirmed" if aa_open else "NOT LB-like -- something didn't take, check above")

  Sanity check Cellulose (LB, unconstrained): 18.3467 mmol/gDW/h
  Sanity check Colanic acid (LB, unconstrained): 3.8691 mmol/gDW/h
  Sanity check PNAG (LB, unconstrained): 18.8616 mmol/gDW/h

Amino acid exchanges open: 20 / 20
LB-like: confirmed


In [7]:
from cobra.io import write_sbml_model

write_sbml_model(model, 'eps_ecoli_model_lb.xml')
print("Saved: eps_ecoli_model_lb.xml")

# Verify the save/reload round-trip fresh, exactly like we did for eps_ecoli_model.xml
lb_model_check = read_sbml_model('eps_ecoli_model_lb.xml')
print(f"\nReloaded fresh -- reactions: {len(lb_model_check.reactions)}, "
      f"metabolites: {len(lb_model_check.metabolites)}")

reloaded_medium = lb_model_check.medium
aa_open_reloaded = [aa for aa in [
    'EX_ala__L_e','EX_arg__L_e','EX_asn__L_e','EX_asp__L_e','EX_cys__L_e',
    'EX_gln__L_e','EX_glu__L_e','EX_gly_e','EX_his__L_e','EX_ile__L_e',
    'EX_leu__L_e','EX_lys__L_e','EX_met__L_e','EX_phe__L_e','EX_pro__L_e',
    'EX_ser__L_e','EX_thr__L_e','EX_trp__L_e','EX_tyr__L_e','EX_val__L_e',
] if aa in reloaded_medium]
print(f"Amino acid exchanges open after fresh reload: {len(aa_open_reloaded)} / 20")

for rxn_id, label in [('EX_cellulose_e', 'Cellulose'), ('EX_colacid_e', 'Colanic acid'), ('EX_puacgam_e', 'PNAG')]:
    with lb_model_check:
        lb_model_check.objective = rxn_id
        sol = lb_model_check.optimize()
        flux = sol.fluxes.get(rxn_id, 0.0)
        print(f"  Reloaded sanity check {label}: {flux:.4f} mmol/gDW/h")

print(f"\nReloaded growth rate: {lb_model_check.slim_optimize():.4f} h⁻¹")

Saved: eps_ecoli_model_lb.xml



Reloaded fresh -- reactions: 2720, metabolites: 1882
Amino acid exchanges open after fresh reload: 20 / 20
  Reloaded sanity check Cellulose: 18.3467 mmol/gDW/h
  Reloaded sanity check Colanic acid: 3.8691 mmol/gDW/h
  Reloaded sanity check PNAG: 18.8616 mmol/gDW/h

Reloaded growth rate: 1.9527 h⁻¹


## Next steps

Two saved model files now cover the two media conventions used throughout the project:
- `eps_ecoli_model.xml` — M9 baked in (`m9_medium_default` convention: only glucose capped at 10, everything else unconstrained)
- `eps_ecoli_model_lb.xml` — LB baked in (same `lb_medium` as `eps_media_comparison.ipynb`: casein hydrolysate amino acids + yeast extract vitamins)

Both are starting points for any new work (dFBA prototype, knockout screen, FVA, etc.) — no need to re-add the three EPS pathways from scratch, and no need to redefine the medium dicts either; just `read_sbml_model(...)` the right file for the medium you need.

Neither file is committed to git (same convention as `iML1515.xml` — should be added to `.gitignore` if not already, since these are large generated files, not source).